In [20]:
import os
import pypdf
import chromadb
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import pickle

In [21]:
# Building the vector database index for the Income Tax PDFs.

PDF_FOLDER   = "data/pdfs"      
DB_FOLDER    = "vector_db"
CHUNK_SIZE   = 400
CHUNK_OVERLAP= 50                # overlap between chunks (helps context)


def extract_text_from_pdf(pdf_path):
    pages = []
    reader = pypdf.PdfReader(pdf_path)
    for page_num, page in enumerate(reader.pages):
        text = page.extract_text()
        if text and text.strip():   # skip blank pages
            pages.append((page_num + 1, text))
    return pages #returns a list of tuples: (page_number, page_text)


def split_into_chunks(text, chunk_size, overlap):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap   # move forward, but keep some overlap
    return chunks


In [22]:
if not os.path.exists(PDF_FOLDER):
    print(f"ERROR: Folder '{PDF_FOLDER}' not found.")
    raise SystemExit(1)

pdf_files = [f for f in os.listdir(PDF_FOLDER) if f.lower().endswith(".pdf")]
if not pdf_files:
    print(f"ERROR: No PDF files found in '{PDF_FOLDER}'.")
    raise SystemExit(1)

print(f"Found {len(pdf_files)} PDF file(s): {pdf_files}\n")

# embedding model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Model loaded\n")

# Set up FAISS(vector database)
dimension = 384  # dimension of the embeddings from all-MiniLM-L6-v2
index = faiss.IndexFlatIP(dimension)  # Inner product for cosine similarity

all_chunks = []
all_ids = []   # a unique ID for each chunk
all_metadata = []   # source file + page number for each chunk

chunk_id = 0

for pdf_file in pdf_files:
    pdf_path = os.path.join(PDF_FOLDER, pdf_file)
    pages = extract_text_from_pdf(pdf_path)

    for page_num, page_text in pages:
        chunks = split_into_chunks(page_text, CHUNK_SIZE, CHUNK_OVERLAP)
        for chunk in chunks:
            if len(chunk.strip()) < 30:  # skip tiny/empty chunks
                continue

            all_chunks.append(chunk)
            all_ids.append(f"chunk_{chunk_id}")
            all_metadata.append({
                "source": pdf_file,
                "page": page_num,
            })
            chunk_id += 1

# Embed all chunks at once
print(f"Embedding {len(all_chunks)} chunks...")
all_embeddings = model.encode(all_chunks, show_progress_bar=True).tolist()

print("\nSaving to vector database...")
# Add to FAISS index
embeddings_np = np.array(all_embeddings).astype('float32')
index.add(embeddings_np)

# Save the index and metadata
faiss.write_index(index, os.path.join(DB_FOLDER, "faiss_index.idx"))
with open(os.path.join(DB_FOLDER, "chunks.pkl"), "wb") as f:
    pickle.dump(all_chunks, f)
with open(os.path.join(DB_FOLDER, "metadata.pkl"), "wb") as f:
    pickle.dump(all_metadata, f)

print(f"Total chunks indexed: {len(all_chunks)}")

Found 12 PDF file(s): ['abcoftax.pdf', 'CBDT_e-Filing_ITR 1_Validation Rules_AY 2025-26_V1.1.pdf', 'CBDT_e-Filing_ITR 4_Validation Rules_AY 2025-26_V1.1.pdf', 'CBDT_e-Filing_ITR 5_Validation Rules_V 1.0.pdf', 'CBDT_e-filing_ITR-3_Validation Rules_V1.0_AY 25-26.pdf', 'CBDT_e-Filing_ITR-7_Validation Rules_V 1.0_AY 25-26.pdf', 'CBDT__e-Filing_ITR 2_Validation Rules_AY 2025-26_V1.0.pdf', 'CBDT__e-Filing_ITR-6_Validation Rules_Version 1.0 (1).pdf', 'generalfaqs.pdf', 'generalinstructions.pdf', 'itr1faqs.pdf', 'usermanual.pdf']



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4115.86it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded

Embedding 2810 chunks...


Batches: 100%|██████████| 88/88 [01:12<00:00,  1.21it/s]



Saving to vector database...
Total chunks indexed: 2810
